# PaperMask v2 — Kaggle training

Predicts the four corners of a photographed document (TL, TR, BR, BL), then perspective-warps the page flat.
One model, one ONNX file: **MobileNetV3-Small → Lite-FPN → heatmap (4×64×64) + sub-pixel offset head**.

**Before you run anything**

1. **Add Input → Datasets →** `walidmostafa77/nexadoc-50k` (7.8 GB; Kaggle extracts it to `/kaggle/input/nexadoc-50k`).
2. Accelerator: **GPU T4 ×2** (P100 also works, but then set `amp: false`). **Internet: on** — the repo is cloned from GitHub and pretrained weights are downloaded.
3. Run the cells top to bottom. Sections 1–3 must succeed before training.

| Step | What it does | Time |
|---|---|---|
| 1. Setup | clone repo, install the 3 packages Kaggle lacks | ~45 s |
| 2. Config | patch paths/batch size into `configs/kaggle_paper.yaml` | instant |
| 3. Preflight | validate 16k labels + images, fail before spending GPU hours | ~40 s |
| 4. Train | ~120 epochs, checkpoints every epoch to `/kaggle/working` | 1–2 h |
| 5. Export | `best.pth` → `paper_v2.onnx` + `model.json`, parity checked | ~10 s |
| 6. Evaluate | test split metrics table | ~1 min |
| 7. Package | zips + download links | instant |

In [ ]:
import os
import shutil
import warnings

# --- 1. SILENCE THE LOGS ---
# TensorFlow is preinstalled on Kaggle and spams cuDNN/C++ warnings even when unused.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

print('✅ Environment silenced.')

# --- 2. SYSTEM CLEANUP + DEPENDENCIES ---
print('⚙️  Setting up environment (~45 s)...')

# TensorFlow competes with PyTorch for GPU memory. It is never imported here.
os.system('pip uninstall -y tensorflow tensorflow-cpu > /dev/null 2>&1')

# protobuf pin fixes Kaggle's TensorBoard 'Red Wall'; albumentations/onnx are not preinstalled.
os.system("pip install -q 'protobuf<4' 'tensorboard>=2.14' albumentations onnx onnxruntime > /dev/null 2>&1")

# --- 3. CLONE THE CODE ---
REPO_URL    = 'https://github.com/walidmoustafa2077/PaperMask.git'
REPO_BRANCH = 'main'          # during development: 'feat/litefpn-heatmap-offset'
REPO_DIR    = '/kaggle/working/PaperMask'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)          # idempotent: safe to re-run after a restart

print(f'📥 Cloning {REPO_BRANCH} → {REPO_DIR}...')
rc = os.system(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
assert rc == 0, 'git clone failed — is the repo public, and is Internet enabled?'

print('✅ Setup complete.')

**Why it looks like this**

* `TF_CPP_MIN_LOG_LEVEL=3` — TensorFlow registers its own cuDNN at import time; the warnings are noise and bury the progress bar.
* `pip uninstall tensorflow` — both frameworks reserve GPU memory. On a 16 GB T4 that is the difference between batch 64 and an OOM.
* `protobuf<4` — Kaggle's protobuf is newer than TensorBoard expects; without the pin the logs are unreadable.
* `shutil.rmtree` before cloning — makes the cell **idempotent**, so you can re-run it after a session restart without a merge error.
* `--depth 1` — we never need history on the VM, and it keeps the clone at ~150 KB.

In [ ]:
import os
import yaml

# --- 1. KAGGLE PATHS ---
DATA_ROOT   = '/kaggle/input/nexadoc-50k'
WORKING_DIR = '/kaggle/working'
RUN_DIR     = f'{WORKING_DIR}/runs/papermask'

%cd /kaggle/working/PaperMask

# --- 2. LOAD THE REPO CONFIG ---
config_path = 'configs/paper_base.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)
print(f'🔄 Loaded {config_path}')

# --- 3. PATCH FOR THIS VM ---
# a. data: Kaggle mounts datasets read-only, always under /kaggle/input.
assert os.path.isdir(DATA_ROOT), f'Dataset missing: add walidmostafa77/nexadoc-50k as Input'
config['data']['root'] = DATA_ROOT

# b. outputs: /kaggle/working is the only writable location, and it is wiped when the session ends.
config['train']['output_dir'] = RUN_DIR
os.makedirs(RUN_DIR, exist_ok=True)

# c. hardware: T4 16 GB. 256×256 + a 1.1 M-param model fits batch 64 comfortably.
config['train']['batch_size']  = 64
config['train']['num_workers'] = 4
config['train']['epochs']      = 120
config['train']['lr']          = 6.0e-4    # 3e-4 was tuned for batch 32
config['train']['amp']         = True      # T4 supports fp16 tensor cores; set False on P100

# --- 4. WRITE THE KAGGLE COPY ---
kaggle_config_path = 'configs/kaggle_paper.yaml'
with open(kaggle_config_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(f'🚀 Wrote {kaggle_config_path}')
print(f'   data.root  = {config["data"]["root"]}')
print(f'   output_dir = {config["train"]["output_dir"]}')
print(f'   batch/epochs/lr = {config["train"]["batch_size"]} / {config["train"]["epochs"]} / {config["train"]["lr"]}')

**Why patch the YAML instead of passing flags**

The repo config is the single source of truth for the architecture — `fpn_channels`, `heatmap_size`, `sigma`. Only the three environment-specific values (dataset root, output directory, batch size) differ on Kaggle. Patching a copy keeps the architecture byte-identical between local runs and the cloud, and the strict config schema rejects any key the trainer does not implement, so a typo fails here instead of 40 minutes into training.

`/kaggle/working` is wiped when the session ends. Everything that must survive is written there and zipped in the last cell.

In [ ]:
%cd /kaggle/working/PaperMask

# Validates dataset layout, label schema, quad geometry and image readability, then exits.
# Deliberately does this BEFORE training: a bad label file would otherwise surface as a
# silently degraded loss curve an hour later.
!python -W ignore src/train.py --config configs/kaggle_paper.yaml --preflight

**Why this cell exists**

The dataset ships as a 7.8 GB archive with 325k files, and its `labels.json` schema is not versioned. The preflight checks:

* every label key resolves to a readable image (and reports up to five offenders);
* every quad has 4 distinct points, positive area, and coordinates in `[0,1]`;
* the JSON is a flat `{filename: [[x,y]×4]}` map — this is what rejects the legacy `[{"value": …, "Count": 2}]` shape;
* the three splits exist with their expected sizes.

It is cheap (~40 s) and it is the difference between "the run is broken" and "the run is 4 hours in and the loss never moved".

### 4. Train

One deliberate difference from the DocShadow notebook: **there is no pre-resize cache step here.**

DocShadow caches because it trains on ~1240×1754 inputs, where JPEG decode and resize dominate the step time (3.0 s/it → 1.2 s/it). Those images are 256×256 by construction — the generator renders, crops and resizes them before writing the JPEG. Decoding one costs ~0.2 ms, roughly 4 % of a batched step, and caching 16k images would add 600 MB of disk writes plus a copy we would have to invalidate on every augmentation change.

If a future experiment moves to a larger input size, add the cache **then** — not before.

In [ ]:
%cd /kaggle/working/PaperMask

print('🚀 Starting training...')

# -W ignore hides the torch.amp deprecation warnings so the metric line stays readable.
# Best checkpoint: /kaggle/working/runs/papermask/best.pth (selected on val MAE).
!python -W ignore src/train.py --config configs/kaggle_paper.yaml

### 5. Resume after a disconnect

Kaggle kills a session at 9–12 h (30 h/week across sessions). Checkpoints are written every epoch, so a crash costs at most one epoch.

Re-run **cells 1 and 2** (they are idempotent — the clone is recreated, the config is rewritten), then:

```python
!python -W ignore src/train.py --config configs/kaggle_paper.yaml \
    --resume /kaggle/working/runs/papermask/last.pth
```

The checkpoint carries `arch_version`, the full config, and the epoch/metrics, so the resume path validates that the architecture has not changed before loading the weights.

In [ ]:
%cd /kaggle/working/PaperMask

import os
CKPT   = '/kaggle/working/runs/papermask/best.pth'
ARTDIR = '/kaggle/working/artifacts'
assert os.path.exists(CKPT), f'No checkpoint at {CKPT} — did training finish?'

# Rebuilds the model from checkpoint['config'], exports opset 17, runs an onnxruntime
# parity check against PyTorch, and writes model.json next to the .onnx.
!python -W ignore src/export.py --checkpoint {CKPT} --output-dir {ARTDIR}

In [ ]:
%cd /kaggle/working/PaperMask

import os
CKPT = '/kaggle/working/runs/papermask/best.pth'

# Full metric table on the held-out test split: MAE/RMSE in pixels, PCK@1/2/5%,
# quad IoU, per-corner breakdown, and the share of invalid quads.
!python -W ignore src/evaluate.py --config configs/kaggle_paper.yaml --checkpoint {CKPT} --split test

In [ ]:
import os
import shutil
import zipfile
from IPython.display import FileLink, display

WORKING = '/kaggle/working'
ARTDIR  = f'{WORKING}/artifacts'
SAMPLE  = f'{WORKING}/runs/papermask/visualizations'

for f in os.listdir(WORKING):
    if f.endswith('.zip'):
        os.remove(os.path.join(WORKING, f))

os.chdir(WORKING)
print(f'📍 {os.getcwd()}')

def zip_dir(src, name):
    if not os.path.isdir(src):
        print(f'⚠️  skipping {name}: {src} not found')
        return
    shutil.make_archive(name, 'zip', src)
    display(FileLink(f'{name}.zip'))
    print(f'✅ {name}.zip  ({os.path.getsize(name + ".zip") / 1e6:.1f} MB)')

def zip_best_only():
    """Only best.pth — the ONNX artifacts are small, epoch checkpoints are not."""
    out = 'papermask_best.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(f'{WORKING}/runs/papermask'):
            for f in files:
                if f == 'best.pth':
                    path = os.path.join(root, f)
                    zf.write(path, os.path.relpath(path, WORKING))
                    print('   →', os.path.relpath(path, WORKING))
    display(FileLink(out))

# 1. The deployable artifacts: .onnx + model.json (+ any parity report)
zip_dir(ARTDIR, 'papermask_onnx')

# 2. Training curves and prediction grids
zip_dir(SAMPLE, 'papermask_samples')

# 3. The checkpoint, in case you want to fine-tune later
zip_best_only()

print('\n⚠️  /kaggle/working is wiped when this session ends — download the three zips now.')

### Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `git clone` exits 128 | repo private, or Internet disabled | set the repo public, or use a PAT: `https://<token>@github.com/...` |
| `ConfigError: unknown key ...` | notebook patched a key the trainer does not implement | the config schema is strict on purpose — remove the key or implement it |
| `AssertionError: Dataset missing` | dataset not attached | Add Input → `walidmostafa77/nexadoc-50k` |
| CUDA out of memory at batch 64 | 2×T4 with a `DataParallel` wrapper doubles peak memory | set `batch_size: 32` in cell 2 |
| `PreflightError: legacy label shape` | dataset version predates the 4-corner contract | report the printed sample; a small adapter converts `[{"value": …}]` → corners |
| Loss flat at ~0.02 and MAE ~40 px | heatmap bias never learned to be negative | check the focal bias init (`-4.6`) survived in `paper_corner.py` |
| ONNX parity check fails | export ran on a different device/dtype than the reference | export with `model.eval()` and fp32; AMP is training-only |

**What to download:** `papermask_onnx.zip` is the only thing the .NET app needs. `papermask_best.zip` is for resuming or fine-tuning. `papermask_samples.zip` is for inspecting corner quality by eye.